In [ ]:
# Modules principaux
import pandas as pd
import json

# Options Pandas
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option("display.max_colwidth", None)
pd.set_option('display.width', 2000)
pd.set_option('display.max_columns', 200)

## 1. Construction du tableau de bord - politique du logement

In [ ]:
countries = [
    "Austria",
    "Belgium",
    "Bulgaria",
    "Croatia",
    "Cyprus",
    "Czechia",
    "Denmark",
    "Estonia",
    "Finland",
    "France",
    "Germany",
    "Greece",
    "Hungary",
    "Ireland",
    "Italy",
    "Latvia",
    "Lithuania",
    "Luxembourg",
    "Malta",
    "Netherlands",
    "Poland",
    "Portugal",
    "Romania",
    "Slovakia",
    "Slovenia",
    "Spain",
    "Sweden"
]

In [ ]:
stats_logement_countries = []

for country in countries:

    with open(f"../data/raw/housing_par_pays/housing_data_{country}.json", "r") as f:
        data = json.load(f)

    public_spending = data["General Housing System"].get("Public Spending", {}).get("content", {})

    completions = data.get("General Housing System", {}).get("Housing Completions", {}).get("content", {}).get("Latest Value")
    population = data.get("General Housing System", {}).get("Population", {}).get("content", {}).get("Latest Value")

    if completions is not None and population is not None:
        annual_completions_1000_hab = completions / population * 1000
    else:
        annual_completions_1000_hab = None

    social_housing_content = data["Affordability"].get("Social Housing", {}).get("content", [])
    if social_housing_content:
        social_housing_stock = social_housing_content[0].get("details", {}).get("Stock %")
    else:
        social_housing_stock = None

    mortgage = data["Affordability"]["Mortgage Market"]["content"][0]["details"]

    emprunt_taux_variables = mortgage.get("% of lending with variable interest rate")
    taux_moyen_nouvel_emprunt = mortgage.get("Avg rate on new loans %")
    montant_moyen_emprunt = mortgage.get("Avg mortgage amount")
    ratio_encours_prets_revenudispo = mortgage.get("Outstanding loans to disposable income ratio %")
    quotite_financement_type = mortgage.get("Typical LTV ratio %")
    maturite_moyenne_prets = mortgage.get("Typical/avg maturity years")

    stats_logements_country = {
        "country": country,
        **public_spending,
        "annual_completions_1000_hab" : annual_completions_1000_hab,
        "social_housing_stock" : social_housing_stock,
        "emprunt_taux_variables" : emprunt_taux_variables,
        "taux_moyen_nouvel_emprunt" : taux_moyen_nouvel_emprunt,
        "montant_moyen_emprunt" : montant_moyen_emprunt,
        "ratio_encours_prets_revenudispo": ratio_encours_prets_revenudispo,
        "quotite_financement_type": quotite_financement_type,
        "maturite_moyenne_prets": maturite_moyenne_prets,    
    }

    stats_logement_countries.append(stats_logements_country)

df_stats_logements_countries = pd.DataFrame(stats_logement_countries)

In [ ]:
df_stats_logements_countries["rent subsidies/housing allowances"] = pd.to_numeric(df_stats_logements_countries["rent subsidies/housing allowances"], errors="coerce")
df_stats_logements_countries["social housing"] = pd.to_numeric(df_stats_logements_countries["social housing"], errors="coerce")
df_stats_logements_countries["homebuying subsidies"] = pd.to_numeric(df_stats_logements_countries["homebuying subsidies"], errors="coerce")
df_stats_logements_countries["renovation subsidies"] = pd.to_numeric(df_stats_logements_countries["renovation subsidies"], errors="coerce")

In [ ]:
colonnes_spending_PIB = [
    "housing development",
    "housing as social protection",
    "rent subsidies/housing allowances",
    "social housing",
    "homebuying subsidies",
    "renovation subsidies"
]

df_stats_logements_countries["public_spending_total"] = df_stats_logements_countries[colonnes_spending_PIB].fillna(0).sum(axis=1)

In [ ]:
df_stats_logements_countries["social_housing_stock"] = pd.to_numeric(df_stats_logements_countries["social_housing_stock"], errors="coerce") * 100
df_stats_logements_countries["taux_moyen_nouvel_emprunt"] = pd.to_numeric(df_stats_logements_countries["taux_moyen_nouvel_emprunt"], errors="coerce")

In [ ]:
df_stats_logements_countries["montant_moyen_emprunt"] = df_stats_logements_countries["montant_moyen_emprunt"].str.replace("BGN ", "").str.replace(",", "")
df_stats_logements_countries["montant_moyen_emprunt"] = pd.to_numeric(df_stats_logements_countries["montant_moyen_emprunt"], errors="coerce")
df_stats_logements_countries["montant_moyen_emprunt"][2] = df_stats_logements_countries["montant_moyen_emprunt"][2] / 1.95583  # convertir les lev bulgares en euros au taux de change indiqué sur le site de la BCE

In [ ]:
df_stats_logements_countries_clean = df_stats_logements_countries[['country','public_spending_total', 'annual_completions_1000_hab', 'social_housing_stock', 'taux_moyen_nouvel_emprunt', 'montant_moyen_emprunt', 'quotite_financement_type', 'maturite_moyenne_prets']].copy()

In [ ]:
df_stats_logements_countries_clean = df_stats_logements_countries_clean.rename(columns={
    "country" : "Pays",
    "public_spending_total" : "Dépenses publiques dans le logement (en % du PIB)",
    "annual_completions_1000_hab" : "Nouveaux logements achevés (pour 1 000 habitants)",
    "social_housing_stock" : "Part du logement social dans le parc de logements",
    "taux_moyen_nouvel_emprunt" : "Taux moyen des nouveaux prêts immobiliers",
    "montant_moyen_emprunt" : "Montant moyen des nouveaux prêts immobiliers",
    "quotite_financement_type" : "Quotité de financement moyenne des prêts immobiliers",
    "maturite_moyenne_prets" : "Maturité moyenne des prêts immobiliers (en années)"
})

In [ ]:
def convertir_fourchette(valeur):

    if pd.isna(valeur):
        return None
    valeur = str(valeur).strip()

    if valeur == "NA" or valeur == "":
        return None
    
    if "-" in valeur:
        parties = valeur.split("-")
        return (float(parties[0]) + float(parties[1])) / 2
    
    else:
        return float(valeur)

In [ ]:
df_stats_logements_countries_clean["Quotité de financement moyenne des prêts immobiliers"] = df_stats_logements_countries_clean["Quotité de financement moyenne des prêts immobiliers"].apply(convertir_fourchette)
df_stats_logements_countries_clean["Maturité moyenne des prêts immobiliers (en années)"] = df_stats_logements_countries_clean["Maturité moyenne des prêts immobiliers (en années)"].apply(convertir_fourchette)

In [ ]:
mapping_pays_datawrapper = {
    "Austria": ":at: Autriche",
    "Belgium": ":be: Belgique",
    "Bulgaria": ":bg: Bulgarie",
    "Cyprus": ":cy: Chypre",
    "Czechia": ":cz: Tchéquie",
    "Germany": ":de: Allemagne",
    "Denmark": ":dk: Danemark",
    "Estonia": ":ee: Estonie",
    "Greece": ":gr: Grèce",
    "Spain": ":es: Espagne",
    "Finland": ":fi: Finlande",
    "France": ":fr: France",
    "Croatia": ":hr: Croatie",
    "Hungary": ":hu: Hongrie",
    "Ireland": ":ie: Irlande",
    "Italy": ":it: Italie",
    "Lithuania": ":lt: Lituanie",
    "Luxembourg": ":lu: Luxembourg",
    "Latvia": ":lv: Lettonie",
    "Malta": ":mt: Malte",
    "Netherlands": ":nl: Pays-Bas",
    "Poland": ":pl: Pologne",
    "Portugal": ":pt: Portugal",
    "Romania": ":ro: Roumanie",
    "Sweden": ":se: Suède",
    "Slovenia": ":si: Slovénie",
    "Slovakia": ":sk: Slovaquie",
}

In [ ]:
df_stats_logements_countries_clean["Pays"] = df_stats_logements_countries_clean["Pays"].map(mapping_pays_datawrapper)

In [ ]:
df_stats_logements_countries_clean = df_stats_logements_countries_clean.sort_values("Pays")

In [ ]:
df_stats_logements_countries_clean.to_excel("../data/clean/politiques_logement_ue_clean.xlsx", index=False)

## 2. Construction du scatter plot - croisement surcharge prix du logement et impayés

In [ ]:
mapping_pays_en_fr = {
    "Austria": "Autriche",
    "Belgium": "Belgique",
    "Bulgaria": "Bulgarie",
    "Cyprus": "Chypre",
    "Czechia": "Tchéquie",
    "Germany": "Allemagne",
    "Denmark": "Danemark",
    "Estonia": "Estonie",
    "Greece": "Grèce",
    "Spain": "Espagne",
    "Finland": "Finlande",
    "France": "France",
    "Croatia": "Croatie",
    "Hungary": "Hongrie",
    "Ireland": "Irlande",
    "Italy": "Italie",
    "Lithuania": "Lituanie",
    "Luxembourg": "Luxembourg",
    "Latvia": "Lettonie",
    "Malta": "Malte",
    "Netherlands": "Pays-Bas",
    "Poland": "Pologne",
    "Portugal": "Portugal",
    "Romania": "Roumanie",
    "Sweden": "Suède",
    "Slovenia": "Slovénie",
    "Slovakia": "Slovaquie"
}

In [ ]:
housing_overburden = pd.read_excel("../data/raw/housing_overburden_2025_raw.xlsx", skiprows=10)

In [ ]:
housing_overburden = housing_overburden.rename(columns={"TIME" : "Pays", "2025" : "Taux de surcharge des coûts du logement*, en %"})
housing_overburden["Pays"] = housing_overburden["Pays"].map(mapping_pays_en_fr)

In [ ]:
retard_paiement = pd.read_excel("../data/raw/retard_paiement_2025_raw.xlsx", skiprows=10)

In [ ]:
retard_paiement = retard_paiement.rename(columns={"GEO (Labels)" : "Pays", "Unnamed: 1" : "Taux de retards de paiement (loyer ou prêt), en %"})
retard_paiement["Pays"] = retard_paiement["Pays"].map(mapping_pays_en_fr)

In [ ]:
df_scatter_logement = pd.merge(housing_overburden, retard_paiement, on="Pays", how="inner")

In [ ]:
df_complet = df_scatter_logement.copy()
df_complet["filtre"] = "Toute l'UE"
df_complet = df_complet.sort_values("Pays")

df_sans_grece = df_scatter_logement[df_scatter_logement["Pays"] != "Grèce"].copy()
df_sans_grece["filtre"] = "Sans la Grèce"
df_sans_grece = df_sans_grece.sort_values("Pays")

df_scatter_logement_final = pd.concat([df_complet, df_sans_grece], ignore_index=True)

In [ ]:
df_scatter_logement_final.to_csv("../data/clean/scatter_logement_clean.csv", index=False)

## 3. Construction de la carte sur la température du logement

In [ ]:
logement_chaud_adequat = pd.read_excel("../data/raw/logement_chaud_adequat_raw.xlsx", skiprows=10)

In [ ]:
logement_chaud_adequat = logement_chaud_adequat.rename(columns={"GEO (Labels)" : "Pays", "Unnamed: 1" : "Incapacité à maintenir une température adéquate dans le logement (en %)"})
logement_chaud_adequat["Pays"] = logement_chaud_adequat["Pays"].map(mapping_pays_en_fr)
logement_chaud_adequat = logement_chaud_adequat.sort_values("Pays")

In [ ]:
logement_chaud_adequat.to_csv("../data/clean/logement_chaud_adequat_clean.csv", sep=",", encoding="utf-8-sig", index=False)